In [ ]:
import numpy as np
from lsst.daf.butler import Butler
from matplotlib import pyplot as plt
%matplotlib inline

## Load some donut stamps and zernikes

In [ ]:
twenty_donut_collection = 'u/brycek/aos_cwfs_step2/danish_dense/wep_v14_13_1/donut_viz_v2_0_4/20250811_20250817'
butler = Butler('/repo/embargo')

In [ ]:
camera = butler.get('camera', dataId={'instrument': 'LSSTCam'}, collections=twenty_donut_collection)

In [ ]:
camera_id_map = camera.getIdMap()

In [ ]:
extra_detector_id = 191
extra_detector_name = camera_id_map[extra_detector_id].getName()
data_id_1 = {'instrument': 'LSSTCam', 'detector': extra_detector_id, 'visit': 2025081300412, 'exposure': 2025081300412}
data_id_1_intra = {'instrument': 'LSSTCam', 'detector': extra_detector_id, 'visit': 2025081300412, 'exposure': 2025081300412}

In [ ]:
ds_intra = butler.get('donutStampsIntra', dataId=data_id_1, collections=twenty_donut_collection)
ds_extra = butler.get('donutStampsExtra', dataId=data_id_1, collections=twenty_donut_collection)
zernikes = butler.get('zernikes', dataId=data_id_1, collections=twenty_donut_collection)

In [ ]:
donut_idx = 0

In [ ]:
plt.imshow(ds_extra[donut_idx].stamp_im.image.array)
plt.title(f'Extra-focal: Idx={donut_idx}')

In [ ]:
plt.imshow(ds_intra[donut_idx].stamp_im.image.array)
plt.title(f'Intra-focal: Idx={donut_idx}')

## Pull out Danish

### Load configurations from `ts_wep`

In [ ]:
from lsst.ts.wep.task import CalcZernikesTask, CalcZernikesTaskConfig, EstimateZernikesDanishTask, DonutStamps
from lsst.ts.wep.utils import getTaskInstrument
from lsst.ts.wep.estimation import WfEstimator

In [ ]:
config = CalcZernikesTaskConfig()

In [ ]:
config.estimateZernikes.retarget(EstimateZernikesDanishTask)

In [ ]:
## Current ts_wep danish pipeline

## These are more about selecting which donuts to use
config.donutStampSelector.maxSelect = 20
config.donutStampSelector.maxFracBadPixels = 2.0e-4
config.donutStampSelector.useCustomSnLimit = True
config.donutStampSelector.minSignalToNoise = 100

## These are options that affect the fitting inside danish
binFactor = 2
config.estimateZernikes.binning = binFactor
nollIndices = np.arange(4, 29)
config.estimateZernikes.nollIndices = list(nollIndices)
config.estimateZernikes.lstsqKwargs = {'ftol': 1.0e-3, 'xtol': 1.0e-3, 'gtol': 1.0e-3}

## This will give more information on the models inside each iteration of the fit.
config.estimateZernikes.saveHistory = False

In [ ]:
task = CalcZernikesTask(config=config)

#### Prepare the donut images for danish

We use the structure that is in `ts_wep/estimation/danish` found [here](https://github.com/lsst-ts/ts_wep/blob/develop/python/lsst/ts/wep/estimation/danish.py).

In [ ]:
camName = 'LSSTCam'
detectorName = extra_detector_name
instrument = getTaskInstrument(
    camName,
    detectorName,
    task.estimateZernikes.config.instConfigFile,
)

# Create the wavefront estimator
wfEst = WfEstimator(
    algoName=task.estimateZernikes.wfAlgoName,
    algoConfig=task.estimateZernikes.wfAlgoConfig,
    instConfig=instrument,
    nollIndices=task.estimateZernikes.config.nollIndices,
    startWithIntrinsic=task.estimateZernikes.config.startWithIntrinsic,
    returnWfDev=task.estimateZernikes.config.returnWfDev,
    units="um",
    saveHistory=task.estimateZernikes.config.saveHistory,
)

In [ ]:
ds_extra_single = DonutStamps(ds_extra[donut_idx:donut_idx+1])
ds_intra_single = DonutStamps(ds_intra[donut_idx:donut_idx+1])

In [ ]:
I1 = ds_extra_single[0].wep_im
I2 = ds_intra_single[0].wep_im

In [ ]:
zkIntrinsicI1 = instrument.getIntrinsicZernikes(
    *I1.fieldAngle,
    I1.bandLabel,
    nollIndices,
)
zkIntrinsicI2 = instrument.getIntrinsicZernikes(
    *I2.fieldAngle,
    I2.bandLabel,
    nollIndices,
)

#### Set up danish configuration with the images

Start from the intrinsic Zernikes

In [ ]:
zkStartI1 = zkIntrinsicI1
zkStartI2 = zkIntrinsicI2

Or start from the initial output from the butler

In [ ]:
from astropy import units as u
zkButler = [zernikes[donut_idx+1][f'Z{z_num}'].to(u.m).value for z_num in nollIndices]

In [ ]:
# zkStartI1 = np.array(zkButler)
# zkStartI2 = np.array(zkButler)

Run the danish code from `ts_wep`

In [ ]:
import danish
import numpy as np
factory = danish.DonutFactory(
    R_outer=instrument.radius,
    R_inner=instrument.radius * instrument.obscuration,
    mask_params=instrument.maskParams,
    focal_length=instrument.focalLength,
    pixel_scale=instrument.pixelSize * task.config.estimateZernikes.binning,
)

In [ ]:
# Prep quantities for both images
img1, angle1, zkRef1, backgroundStd1 = wfEst.algo._prepDanish(
    image=I1,
    zkStart=zkStartI1,
    nollIndices=nollIndices,
    instrument=instrument,
)
img2, angle2, zkRef2, backgroundStd2 = wfEst.algo._prepDanish(
    image=I2,
    zkStart=zkStartI2,
    nollIndices=nollIndices,
    instrument=instrument,
)

# Package these into lists for Danish
imgs = [img1, img2]
thxs = [angle1[0], angle2[0]]
thys = [angle1[1], angle2[1]]
zkRefs = [zkRef1, zkRef2]
skyLevels = [backgroundStd1**2, backgroundStd2**2]

# Create Double Zernike tuples
dzTerms = [(1, j) for j in nollIndices]

# Set field radius to max value from mask params
fieldRadius = np.deg2rad(
    np.max(
        [
            edge["thetaMax"]
            for item in instrument.maskParams.values()
            for edge in item.values()
        ]
    )
)

# Create model
model = danish.MultiDonutModel(
    factory,
    z_refs=zkRefs,
    dz_terms=dzTerms,
    field_radius=fieldRadius,
    thxs=thxs,
    thys=thys,
    npix=imgs[0].shape[0],
)

In [ ]:
x0 = [0.0] * 2 + [0.0] * 2 + [0.7] + [0.0] * len(dzTerms)

In [ ]:
from scipy.optimize import least_squares

In [ ]:
bounds = [[-np.inf, np.inf]]*(5+len(dzTerms))
bounds[4] = [0.1, 5.0]
bounds = [list(b) for b in zip(*bounds)]

#### Run the optimization to find the danish result

In [ ]:
result = least_squares(
    model.chi,
    jac=model.jac,
    x0=x0,
    args=(imgs, skyLevels),
    bounds=bounds,
    **task.config.estimateZernikes.lstsqKwargs,
)

#### Get the results out

In [ ]:
result = dict(result)

# Unpack the parameters
dxs, dys, fwhm, zkFit = model.unpack_params(result["x"])

# Add the starting zernikes back into the result
zkSum = zkFit + np.nanmean([zkStartI1, zkStartI2], axis=0)

In [ ]:
modelImages = model.model(
    dxs,
    dys,
    fwhm,
    zkFit,
    sky_levels=skyLevels,
    fluxes=np.sum(imgs, axis=(1, 2)),
)

In [ ]:
plt.imshow(modelImages[0])

Unbin model image

In [ ]:
from scipy.ndimage import zoom

In [ ]:
model_unbinned = [zoom(np.array(model_image), binFactor, order=3) for model_image in modelImages]

In [ ]:
plt.imshow(model_unbinned[0])
plt.colorbar()

In [ ]:
plt.imshow(I1.image)
plt.colorbar()

Show residual

In [ ]:
fig = plt.figure(figsize=(18, 6))
fig.add_subplot(2,3,1)
plt.imshow(I1.image[:-binFactor, :-binFactor])
plt.colorbar()
plt.xlabel('X Pixels')
plt.ylabel('Y Pixels')
plt.title('Extra-focal Data')

fig.add_subplot(2,3,2)
plt.imshow(model_unbinned[0])
plt.colorbar()
plt.xlabel('X Pixels')
plt.ylabel('Y Pixels')
plt.title('Unbinned Extra-Focal Model')

fig.add_subplot(2,3,3)
plt.imshow(I1.image[:-binFactor, :-binFactor] - model_unbinned[0], cmap=plt.get_cmap('coolwarm'))
plt.colorbar()
plt.xlabel('X Pixels')
plt.ylabel('Y Pixels')
plt.title('Extra-focal Residual')

fig.add_subplot(2,3,4)
plt.imshow(I2.image[:-binFactor, :-binFactor])
plt.colorbar()
plt.xlabel('X Pixels')
plt.ylabel('Y Pixels')
plt.title('Intra-focal Data')

fig.add_subplot(2,3,5)
plt.imshow(model_unbinned[1])
plt.colorbar()
plt.xlabel('X Pixels')
plt.ylabel('Y Pixels')
plt.title('Unbinned Intra-focal Model')

fig.add_subplot(2,3,6)
plt.imshow(I2.image[:-binFactor, :-binFactor] - model_unbinned[1], cmap=plt.get_cmap('coolwarm'))
plt.colorbar()
plt.xlabel('X Pixels')
plt.ylabel('Y Pixels')
plt.title('Intra-focal Residual')

plt.tight_layout()

In [ ]:
plt.plot(nollIndices, zkSum / 1e-6, label='New Danish fit')
plt.plot(nollIndices, np.array(zkButler) / 1e-6, label='Butler fit')
plt.xlabel('Noll Index')
plt.ylabel('Zernike coefficient (microns)')
plt.legend()

In [ ]:
## "Donut Blur" estimate from fit in arcsec
print(f'New Fit Donut Blur FWHM: {fwhm:.4f} arcsec')
print(f'Butler Fit Donut Blur FWHM: {zernikes.meta['estimatorInfo']['fwhm'][donut_idx]:.4f} arcsec')